# Model 06 — Random Forest (Pharmaceutical Syrup pH Lot Approval)

pH compliance in oral liquid pharmaceutical manufacturing is not a courtesy metric — it is a release criterion. A syrup lot with a final pH outside the 4.5–5.5 specification cannot be released. It goes to reprocess, which means reblending, re-analysis, delay in the batch schedule, and a deviation event that requires investigation. The cost is not just material; it is regulatory throughput and documentation burden.

The challenge is that pH in a syrup formulation is the result of multiple interacting factors: the concentration of citric acid (which drives pH down), the citrate buffer system (which resists pH change), the quality of the purified water entering the tank, the residual pH carried over from the previous lot, and the process conditions that determine whether the formulation actually achieves its intended chemistry — mixing time, agitation speed, and ingredient addition order. No single variable controls the outcome. The pH emerges from the combination.

**Random Forest** is the right model for this problem. Ensemble learning through bagging reduces variance — the core issue with single decision trees on formulation data where observations are noisy and interactions are nonlinear. The forest interrogates the data through hundreds of independent trees and returns a probability estimate that represents the consensus across many possible decision paths. This consensus is more stable and more calibrated than any individual tree.

This project builds a lot approval classifier — a model that predicts whether a batch will achieve in-specification pH before the lot is closed, based on the formulation parameters and process conditions recorded at manufacture time. The output supports both real-time process monitoring and retrospective failure analysis.

---
**Data source:** `rf_raw_data.csv`  
**Output:** metrics, permutation importance, feature importance comparison, and a batch scenario simulator


## Section 1 — Setup

Reproducibility in pharmaceutical data analysis is a regulatory expectation, not just good practice. We fix a global seed so that any re-run of this notebook produces identical results — essential when model outputs are used in process validation documentation.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    roc_curve, roc_auc_score, precision_recall_curve, average_precision_score,
    f1_score, precision_score, recall_score
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11

print("Libraries loaded. Random state:", RANDOM_STATE)


## Section 2 — Load Data

The dataset contains **1,847 pharmaceutical batch records** from a pharmaceutical syrup manufacturing line. Each row represents one lot, described by 11 process and formulation variables recorded at the time of manufacture: active pharmaceutical ingredient concentration, buffer and acid percentages, sweetener and preservative levels, purified water pH, previous lot pH, mixing temperature, mixing time, agitation speed, and ingredient addition order. The target variable `lot_approved` is binary — 1 if the final pH fell within the 4.5–5.5 specification window, 0 if the lot failed pH specification and required reprocessing. The approval rate is **41.4%** (58.6% failed).

The column `final_ph` records the actual measured pH for each lot. It is included in the dataset for reference but is **excluded from model inputs** — the model's purpose is to predict approval outcome from process parameters before the final pH assay result is available. Including `final_ph` as a feature would trivially solve the classification problem but defeat its operational purpose.

In [ ]:
try:
    df = pd.read_csv("rf_raw_data.csv")
except FileNotFoundError:
    df = pd.read_csv("https://raw.githubusercontent.com/LozanoLsa/PH_Adjustment_Syrup/main/rf_raw_data.csv")
    # FileNotFoundError is intentionally specific — a bare except would silently swallow real data errors

FEATURES = [
    "api_pct", "citrate_buffer_pct", "citric_acid_pct", "sweetener_pct",
    "preservative_pct", "water_ph", "prev_lot_ph", "mixing_temp_c",
    "mixing_time_min", "agitation_rpm", "addition_order"
]
TARGET = "lot_approved"

# addition_order encoding: 1=buffer→acid→API, 2=acid→buffer→API, 3=API→buffer→acid
ORDER_LABELS = {1: "Buffer→Acid→API", 2: "Acid→Buffer→API", 3: "API→Buffer→Acid"}

df.head()


## Section 3 — Quick Sanity Checks

Before modeling, we confirm the dataset loaded correctly and inspect the class balance. With a **41.4% lot-approval rate**, this is a moderately imbalanced problem. Precision and Recall for the non-approved class (0) are as important as for the approved class — missing a failing lot has regulatory consequences, and falsely flagging a good lot has production cost implications.

In [ ]:
# Sanity checks. Real datasets usually try to hurt you 🙂
print("Shape:", df.shape)
print("\n--- Data types ---")
df.info()


In [ ]:
print("--- Missing values ---")
print(df.isna().sum())
print("\n--- Lot approval rate ---")
print(df[TARGET].value_counts())
print(df[TARGET].value_counts(normalize=True)
      .rename({0: "not_approved", 1: "approved"}))


In [ ]:
print("--- Final pH distribution ---")
print(df["final_ph"].describe().round(3))
print(f"\nLots within spec (4.5–5.5): {((df['final_ph']>=4.5) & (df['final_ph']<=5.5)).sum()}")
print("\n--- Numeric summary (formulation + process) ---")
df[FEATURES].describe().round(2)


## Section 4 — Exploratory Data Analysis

EDA here is about understanding the chemistry before handing the problem to the model. We want to confirm that the variables with the strongest mechanistic relationship to pH — citric acid and the citrate buffer — show visible separation between approved and non-approved lots. We also want to understand whether process variables like mixing time and agitation speed show any discriminating power, and whether addition order creates systematic differences in pH outcomes.

The histogram of `final_ph` provides context for the classification boundary: how sharp is the 4.5–5.5 cutoff in practice, and how many lots are borderline cases near the specification limits?


In [ ]:
# Final pH distribution — context for the classification boundary
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df["final_ph"], bins=40, color="#4C9BE8", edgecolor="white", alpha=0.8)
ax.axvline(4.5, color="#E8574C", linestyle="--", linewidth=1.8, label="Lower spec limit (4.5)")
ax.axvline(5.5, color="#E8574C", linestyle="--", linewidth=1.8, label="Upper spec limit (5.5)")
ax.set_xlabel("Final pH")
ax.set_ylabel("Count")
ax.set_title("Distribution of Final pH — Lot Release Specification: 4.5–5.5",
             fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()
print(f"In-spec rate: {df[TARGET].mean():.1%}  |  "
      f"Below 4.5: {(df['final_ph']<4.5).mean():.1%}  |  "
      f"Above 5.5: {(df['final_ph']>5.5).mean():.1%}")


In [ ]:
# Formulation variables — distributions by approval status
form_cols = ["citric_acid_pct", "citrate_buffer_pct", "api_pct",
             "sweetener_pct", "preservative_pct"]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()
for i, col in enumerate(form_cols):
    axes[i].hist(df[df[TARGET]==0][col], bins=25, alpha=0.6,
                 color="#E8574C", edgecolor="white", label="Not Approved")
    axes[i].hist(df[df[TARGET]==1][col], bins=25, alpha=0.6,
                 color="#4C9BE8", edgecolor="white", label="Approved")
    axes[i].set_title(col.replace("_", " ").title())
    axes[i].legend(fontsize=8)
axes[-1].set_visible(False)
plt.suptitle("Formulation Variables — Approved vs Not Approved Lots",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# Process variables — distributions by approval status
proc_cols = ["water_ph", "prev_lot_ph", "mixing_temp_c",
             "mixing_time_min", "agitation_rpm"]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()
for i, col in enumerate(proc_cols):
    axes[i].hist(df[df[TARGET]==0][col], bins=25, alpha=0.6,
                 color="#E8574C", edgecolor="white", label="Not Approved")
    axes[i].hist(df[df[TARGET]==1][col], bins=25, alpha=0.6,
                 color="#4C9BE8", edgecolor="white", label="Approved")
    axes[i].set_title(col.replace("_", " ").title())
    axes[i].legend(fontsize=8)
axes[-1].set_visible(False)
plt.suptitle("Process Variables — Approved vs Not Approved Lots",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# Addition order effect — approval rate by each order sequence
order_approval = df.groupby("addition_order")[TARGET].mean()
order_labels = [ORDER_LABELS[k] for k in order_approval.index]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(order_labels, order_approval.values,
              color=["#4C9BE8" if v > df[TARGET].mean() else "#E8574C"
                     for v in order_approval.values],
              edgecolor="white")
ax.axhline(df[TARGET].mean(), linestyle="--", color="gray",
           linewidth=1.2, label=f"Overall approval rate {df[TARGET].mean():.1%}")
for bar, v in zip(bars, order_approval.values):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.005,
            f"{v:.1%}", ha="center", fontweight="bold", fontsize=10)
ax.set_ylabel("Lot Approval Rate")
ax.set_title("Lot Approval Rate by Ingredient Addition Order", fontweight="bold")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
# Correlation heatmap — focused on relationships relevant to pH
fig, ax = plt.subplots(figsize=(11, 8))
corr = df[FEATURES + ["final_ph", TARGET]].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            vmin=-1, vmax=1, linewidths=0.5, ax=ax, annot_kws={"size": 8})
ax.set_title("Feature Correlation Matrix", fontweight="bold")
plt.tight_layout()
plt.show()


## Section 5 — Preprocessing

Random Forest is a tree-based ensemble — it makes decisions based on value thresholds, not distances. This means it does not require feature scaling: a concentration variable measured in percentages and a temperature measured in Celsius compete on equal footing because the tree only asks *"is this value above or below threshold X?"*.

The `addition_order` variable is already integer-encoded (1, 2, 3), which the tree handles natively. There are no categorical variables requiring one-hot encoding. Preprocessing is therefore minimal: we simply separate features from target, drop the non-modeling columns (`lot_id`, `final_ph`), and split with stratification to preserve the **41.4% approval rate** in both sets.

In [ ]:
# Feature matrix — exclude lot_id and final_ph
# final_ph is excluded intentionally: predicting approval from process params,
# not from the pH result itself
X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y
)

print(f"Training set : {X_train.shape[0]} batches")
print(f"Test set     : {X_test.shape[0]} batches")
print(f"Approval rate: train={y_train.mean():.3f}  test={y_test.mean():.3f}")
print(f"Features     : {len(FEATURES)}")
print(f"\nNo scaling required — Random Forest uses threshold splits, not distances.")


## Section 6 — Train Model

We train a Random Forest with 300 trees — a number large enough to stabilize the ensemble variance without significant additional benefit beyond this point. The trees are grown without depth limitation (`max_depth=None`), which allows each tree to fully capture the complex interactions between formulation variables and process conditions. Overfitting at the individual tree level is controlled by the ensemble averaging, not by depth restriction.

For comparison, we also train a single Decision Tree with the same data. This baseline illustrates the core value of ensemble learning: a single tree will typically overfit and generalize poorly on formulation data, while the forest's variance reduction through bagging produces a significantly more stable and accurate model.


In [ ]:
# Random Forest — 300 trees, unconstrained depth, bagging handles variance
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf.fit(X_train, y_train)

# Single Decision Tree — baseline for comparison
dt_baseline = DecisionTreeClassifier(random_state=RANDOM_STATE)
dt_baseline.fit(X_train, y_train)

print("Models trained successfully.")
print(f"Random Forest : {rf.n_estimators} trees | max_depth=None | n_features/split=sqrt(p)")
print(f"Decision Tree : max_depth=None (fully grown, no pruning)")


## Section 7 — Evaluate

In pharmaceutical lot release, both error directions carry real costs:

- **False Negatives** (predicting approval for a lot that will fail pH): the lot proceeds, fails final QC, triggers a deviation investigation, and potentially requires reprocessing after reagents have already been consumed.
- **False Positives** (flagging a conforming lot as likely to fail): unnecessary hold, additional review cycle, and potential delay in batch release schedule.

The AUC of 0.90 confirms that the Random Forest has strong discriminatory power — it ranks failing lots well above passing lots across all decision thresholds. We compare performance against the single Decision Tree baseline to quantify the practical value of ensemble learning on this formulation dataset.


In [ ]:
y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

y_pred_dt = dt_baseline.predict(X_test)

acc_rf = accuracy_score(y_test, y_pred)
auc_rf = roc_auc_score(y_test, y_prob)
acc_dt = accuracy_score(y_test, y_pred_dt)

print("=== Random Forest ===")
print(f"Accuracy  : {acc_rf:.4f}")
print(f"AUC-ROC   : {auc_rf:.4f}")
print(f"Precision (Approved): {precision_score(y_test, y_pred):.4f}")
print(f"Recall    (Approved): {recall_score(y_test, y_pred):.4f}")
print(f"F1        (Approved): {f1_score(y_test, y_pred):.4f}")
print("\n", classification_report(y_test, y_pred, target_names=["Not Approved", "Approved"]))

print("=== Decision Tree (baseline) ===")
print(f"Accuracy  : {acc_dt:.4f}")
print(f"AUC-ROC   : {roc_auc_score(y_test, dt_baseline.predict_proba(X_test)[:,1]):.4f}")
print(f"\nRF improvement over DT: {acc_rf - acc_dt:+.4f} accuracy | "
      f"{auc_rf - roc_auc_score(y_test, dt_baseline.predict_proba(X_test)[:,1]):+.4f} AUC")


In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
labels = ["Not Approved", "Approved"]

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=labels, yticklabels=labels,
            linewidths=0.5, ax=ax)
ax.set_xlabel("Predicted", fontweight="bold")
ax.set_ylabel("Actual", fontweight="bold")
ax.set_title("Confusion Matrix — Random Forest", fontweight="bold")
plt.tight_layout()
plt.show()
print("Rows = actual outcome. Columns = predicted. Diagonal = correct predictions.")


In [ ]:
# ROC + Precision-Recall curves — RF vs DT comparison
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob)
fpr_dt, tpr_dt, _ = roc_curve(y_test, dt_baseline.predict_proba(X_test)[:,1])
prec_c, rec_c, _  = precision_recall_curve(y_test, y_prob)
ap = average_precision_score(y_test, y_prob)
auc_dt = roc_auc_score(y_test, dt_baseline.predict_proba(X_test)[:,1])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(fpr_rf, tpr_rf, color="#4C9BE8", lw=2, label=f"Random Forest AUC={auc_rf:.3f}")
axes[0].plot(fpr_dt, tpr_dt, color="#F0A500", lw=1.5, linestyle="--",
             label=f"Decision Tree AUC={auc_dt:.3f}")
axes[0].plot([0,1],[0,1], linestyle=":", color="gray", lw=1, label="Random baseline")
axes[0].fill_between(fpr_rf, tpr_rf, alpha=0.1, color="#4C9BE8")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC Curve — RF vs Decision Tree", fontweight="bold")
axes[0].legend()

axes[1].plot(rec_c, prec_c, color="#E8574C", lw=2, label=f"RF AP={ap:.3f}")
axes[1].axhline(y_test.mean(), linestyle="--", color="gray", lw=1,
                label=f"Baseline (approval rate={y_test.mean():.2f})")
axes[1].fill_between(rec_c, prec_c, alpha=0.1, color="#E8574C")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curve — Random Forest", fontweight="bold")
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
# Predicted probability distribution by true class
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(y_prob[y_test == 0], bins=30, alpha=0.6, color="#E8574C",
        label="Actual: Not Approved", edgecolor="white")
ax.hist(y_prob[y_test == 1], bins=30, alpha=0.6, color="#4C9BE8",
        label="Actual: Approved", edgecolor="white")
ax.axvline(0.5, color="black", linestyle="--", lw=1.5, label="Decision threshold (0.5)")
ax.set_xlabel("Predicted Approval Probability")
ax.set_ylabel("Count")
ax.set_title("Predicted Probability Distribution by True Class", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()


## Section 8 — Interpretability: Feature Importance

Random Forest provides two complementary importance measures, each answering a slightly different question:

**Gini (built-in) importance** answers: *"how much did this feature contribute to reducing impurity across all trees and all splits?"* Features that appear early in many trees — because they split the data most cleanly — accumulate the highest Gini scores. This measure is fast to compute but can be biased toward high-cardinality continuous variables.

**Permutation importance** answers: *"how much does accuracy drop when this feature's values are randomly shuffled on the test set?"* A large accuracy drop means the model was genuinely relying on that feature for correct predictions on unseen data — not just on training data. This is the more conservative and trustworthy measure for generalization.

Comparing both measures reveals which features are consistently important (high in both) and which may be artifacts of the Gini calculation (high in Gini, low in permutation).


In [ ]:
# Gini importance
gini_df = pd.DataFrame({
    "Feature":    FEATURES,
    "Gini":       rf.feature_importances_
}).sort_values("Gini", ascending=False).reset_index(drop=True)

print("Gini Feature Importance (reduction in impurity):")
print(gini_df.round(4).to_string(index=False))


In [ ]:
# Permutation importance on test set
perm = permutation_importance(
    rf, X_test, y_test,
    n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1
)

perm_df = pd.DataFrame({
    "Feature":    FEATURES,
    "Permutation": perm.importances_mean,
    "Std":         perm.importances_std
}).sort_values("Permutation", ascending=False).reset_index(drop=True)

print("Permutation Importance (accuracy drop when feature is shuffled):")
print(perm_df.round(4).to_string(index=False))


In [ ]:
# Side-by-side comparison — Gini vs Permutation
combined = gini_df.merge(perm_df[["Feature","Permutation"]], on="Feature")
combined = combined.sort_values("Gini", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Gini
gi_sorted = combined.sort_values("Gini", ascending=True)
axes[0].barh(gi_sorted["Feature"].str.replace("_"," ").str.title(),
             gi_sorted["Gini"],
             color=["#E8574C" if v > 0.05 else "#4C9BE8" for v in gi_sorted["Gini"]],
             edgecolor="white")
axes[0].set_xlabel("Gini Importance")
axes[0].set_title("Built-in Importance (Gini)", fontweight="bold")
axes[0].axvline(0.05, color="gray", linestyle="--", linewidth=1)

# Permutation
pm_sorted = combined.sort_values("Permutation", ascending=True)
axes[1].barh(pm_sorted["Feature"].str.replace("_"," ").str.title(),
             pm_sorted["Permutation"],
             color=["#E8574C" if v > 0.01 else "#4C9BE8" for v in pm_sorted["Permutation"]],
             edgecolor="white")
axes[1].set_xlabel("Permutation Importance (accuracy drop)")
axes[1].set_title("Permutation Importance (test set)", fontweight="bold")

plt.suptitle("Feature Importance — Gini vs Permutation Comparison",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


### Summary of Practical Signals

Both importance measures converge on the same story — one that any formulation chemist would recognize:

- **`citric_acid_pct` is the dominant driver (26.4% Gini)** — citric acid is the primary acidulant in the formulation. Excess citric acid pushes pH below 4.5; insufficient acid fails to reach the lower bound from the buffer-dominated baseline. It is the variable with the highest leverage on the final pH outcome.
- **`citrate_buffer_pct` is the counter-lever (20.8%)** — the citrate buffer system resists pH change and stabilizes the formulation toward ~5.0. High buffer concentration with appropriate acid yields in-spec pH; low buffer concentration makes the system hypersensitive to variations in acid and water.
- **Process variables carry real weight**: `agitation_rpm` (7.9%), `prev_lot_ph` (7.7%), `water_ph` (7.1%), and `mixing_time_min` (6.9%) all rank above the minor formulation components. This confirms that in-specification pH is not purely a formulation problem — it is also a process execution problem. A correctly formulated batch run at insufficient agitation can still fail pH.
- **`addition_order` is the least influential variable (1.9%)** — the order effect exists but is smaller than the other process variables, suggesting it acts mainly in interaction with poor mixing rather than as an independent driver.
- **`sweetener_pct` and `api_pct`** contribute small but non-zero importance — likely through indirect pH interactions at the concentrations used in this formulation range.

**Operational implication:** pH robustness improvement should focus first on tight control of citric acid and buffer concentrations, then on ensuring adequate agitation and mixing time. Water pH monitoring is a necessary input check. Addition order is a useful SOP standardization measure but will not compensate for formulation or process variability.


## Section 9 — Statistical Validation

Random Forest is an ensemble model with built-in variance reduction through bagging, which makes it inherently more stable than single-tree classifiers. However, we still validate explicitly with 5-fold stratified cross-validation and a confidence interval for test accuracy — especially because the class imbalance (**41.4% approval rate**) means that minority-class performance estimates can be noisy with a single split.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_acc = cross_val_score(rf, X_train, y_train, cv=cv, scoring="accuracy")
cv_auc = cross_val_score(rf, X_train, y_train, cv=cv, scoring="roc_auc")
cv_f1  = cross_val_score(rf, X_train, y_train, cv=cv, scoring="f1")

print("5-Fold Stratified Cross-Validation (training set only):")
print(f"  Accuracy : {cv_acc.mean():.4f} ± {cv_acc.std():.4f}")
print(f"  AUC-ROC  : {cv_auc.mean():.4f} ± {cv_auc.std():.4f}")
print(f"  F1       : {cv_f1.mean():.4f}  ± {cv_f1.std():.4f}")


In [ ]:
# 95% confidence interval for test accuracy
n = len(y_test)
p = accuracy_score(y_test, y_pred)
z = 1.96
margin = z * np.sqrt((p * (1 - p)) / n)

print(f"Test-set accuracy : {p:.4f}")
print(f"95% CI            : [{p - margin:.4f}, {p + margin:.4f}]")
print(f"Margin of error   : ±{margin:.4f}")
print()
print(f"Train/Test gap    : {accuracy_score(y_train, rf.predict(X_train)):.4f} train "
      f"vs {p:.4f} test = {accuracy_score(y_train, rf.predict(X_train))-p:+.4f}")
print("Note: RF typically shows some train/test gap due to fully grown trees;")
print("the ensemble averaging keeps test performance stable across folds.")


## Section 10 — Batch Scenario Simulator

A simulator converts the trained model into a what-if tool for formulation development and process troubleshooting. Before a batch is manufactured, a formulation chemist or process engineer can enter the planned recipe parameters and process conditions to receive a predicted probability of pH compliance. This enables pre-production risk assessment — identifying which formulation or process adjustments are most likely to move a borderline recipe into specification.

The simulator is also useful for retrospective analysis: entering the parameters of a failed lot allows the model to identify which variables were most responsible for the pH deviation.


In [ ]:
def simulate_batch(
    api_pct: float = 2.0,
    citrate_buffer_pct: float = 3.0,
    citric_acid_pct: float = 0.8,
    sweetener_pct: float = 25.0,
    preservative_pct: float = 0.5,
    water_ph: float = 7.0,
    prev_lot_ph: float = 5.0,
    mixing_temp_c: float = 25.0,
    mixing_time_min: float = 30.0,
    agitation_rpm: float = 500.0,
    addition_order: int = 1,
    threshold: float = 0.5
) -> tuple:
    """
    Predict pH specification compliance for a batch before manufacture.

    Parameters
    ----------
    api_pct             : Active ingredient concentration % (range: 0.5–5.0)
    citrate_buffer_pct  : Citrate buffer % (range: 1.0–5.0; higher = more pH stability)
    citric_acid_pct     : Citric acid % (range: 0.0–2.0; higher = lower pH)
    sweetener_pct       : Sweetener % (range: 5.0–40.0)
    preservative_pct    : Preservative % (range: 0.1–1.0)
    water_ph            : Purified water pH (range: 6.0–7.5)
    prev_lot_ph         : pH of the previous lot in the same vessel (range: 4.0–6.0)
    mixing_temp_c       : Mixing temperature °C (range: 15–35)
    mixing_time_min     : Mixing duration in minutes (range: 5–60)
    agitation_rpm       : Agitation speed RPM (range: 100–800)
    addition_order      : 1=Buffer→Acid→API, 2=Acid→Buffer→API, 3=API→Buffer→Acid
    threshold           : Classification cutoff (default 0.5)
    """
    X_new = pd.DataFrame([{
        "api_pct": api_pct, "citrate_buffer_pct": citrate_buffer_pct,
        "citric_acid_pct": citric_acid_pct, "sweetener_pct": sweetener_pct,
        "preservative_pct": preservative_pct, "water_ph": water_ph,
        "prev_lot_ph": prev_lot_ph, "mixing_temp_c": mixing_temp_c,
        "mixing_time_min": mixing_time_min, "agitation_rpm": agitation_rpm,
        "addition_order": addition_order
    }])

    prob_ok = rf.predict_proba(X_new)[0, 1]
    pred_ok = int(prob_ok >= threshold)

    print("=" * 60)
    print("  BATCH pH COMPLIANCE PREDICTOR")
    print("=" * 60)
    print(f"  API                : {api_pct:.1f}%")
    print(f"  Citrate buffer     : {citrate_buffer_pct:.1f}%")
    print(f"  Citric acid        : {citric_acid_pct:.1f}%")
    print(f"  Sweetener          : {sweetener_pct:.1f}%")
    print(f"  Preservative       : {preservative_pct:.2f}%")
    print(f"  Water pH           : {water_ph:.2f}")
    print(f"  Previous lot pH    : {prev_lot_ph:.2f}")
    print(f"  Mixing temperature : {mixing_temp_c:.1f} °C")
    print(f"  Mixing time        : {mixing_time_min:.0f} min")
    print(f"  Agitation          : {agitation_rpm:.0f} rpm")
    print(f"  Addition order     : {addition_order} ({ORDER_LABELS[addition_order]})")
    print("-" * 60)
    print(f"  P(pH in spec)      : {prob_ok:.3f} ({prob_ok*100:.1f}%)")
    print(f"  Decision threshold : {threshold:.2f}")
    if pred_ok == 1:
        print("  Prediction: >>> APPROVED — pH LIKELY IN SPECIFICATION <<<")
    else:
        print("  Prediction: >>> REVIEW — pH OUT-OF-SPEC RISK ELEVATED <<<")
    print("=" * 60)
    return prob_ok, pred_ok


### Scenario A — Well-formulated batch: high buffer, moderate acid, good process conditions


In [ ]:
simulate_batch(
    api_pct=2.0,
    citrate_buffer_pct=4.0,
    citric_acid_pct=0.8,
    sweetener_pct=30.0,
    preservative_pct=0.5,
    water_ph=7.0,
    prev_lot_ph=5.0,
    mixing_temp_c=25.0,
    mixing_time_min=40.0,
    agitation_rpm=600.0,
    addition_order=1   # Buffer→Acid→API: most stable sequence
)


### Scenario B — High-risk batch: excess acid, low buffer, poor process conditions


In [ ]:
simulate_batch(
    api_pct=2.0,
    citrate_buffer_pct=1.2,  # low buffer — poor pH resistance
    citric_acid_pct=1.7,     # high acid — pH likely below 4.5
    sweetener_pct=20.0,
    preservative_pct=0.8,
    water_ph=6.5,
    prev_lot_ph=4.5,         # previous lot already at lower boundary
    mixing_temp_c=30.0,      # elevated temperature
    mixing_time_min=10.0,    # insufficient mixing time
    agitation_rpm=150.0,     # poor agitation — inadequate homogenization
    addition_order=2         # Acid→Buffer→API: higher risk sequence
)


### Scenario C — What-if: same risk profile, correcting buffer and agitation only


In [ ]:
# Two targeted corrections on the high-risk profile from Scenario B
simulate_batch(
    api_pct=2.0,
    citrate_buffer_pct=3.5,  # corrected: buffer increased
    citric_acid_pct=1.7,
    sweetener_pct=20.0,
    preservative_pct=0.8,
    water_ph=6.5,
    prev_lot_ph=4.5,
    mixing_temp_c=30.0,
    mixing_time_min=10.0,
    agitation_rpm=550.0,     # corrected: agitation increased
    addition_order=2
)


## Final Reflection

This project is not about predicting pH with certainty.  
It is about **making pH deviation risk visible before the batch is manufactured**.

Pharmaceutical syrup pH is the outcome of a system — not a single variable. Citric acid, citrate buffer, purified water quality, previous lot carryover, mixing conditions, and addition sequence all interact to produce the final pH value. These interactions are not linear and they are not obvious from the formulation sheet alone. The model captures them from operational batch history, and makes them available as a pre-production risk score.

What this model offers is not a release decision, but a **conversation**:
- *What happens to pH compliance probability if we increase the buffer from 2.5% to 3.5%?*
- *How much does agitation speed matter when the previous lot was at the lower pH boundary?*
- *Which batches in this campaign are most likely to require pH adjustment before filling?*

These questions matter more than the AUC score.

---

*LozanoLsa  |  Operational Excellence · MBB · Machine Learning  |  GitHub: LozanoLsa*
